# Stage A - Multilingual Transformer Benchmarking & Model Selection
**Component 3:** Bilingual NLP, Emotion-Aware Modelling, and Wellbeing Indicator Classification  
**Owner:** Ramanayake R. H. B. D. G. (IT23164130)  

### Objective:
Fine-tune and benchmark candidate multilingual transformer models:
1. `xlm-roberta-base`
2. `bert-base-multilingual-cased` (mBERT)

Compare results against the TF-IDF performance floors on the **exact same 20% test splits** for:
- **Dreaddit** (Stress detection)
- **GoEmotions** (Emotion classification)

The winning candidate is selected as the multilingual backbone for **Stage B** (Domain-Adaptive Pretraining & Multi-task Architecture).

In [ ]:
# 1. Install required packages
!pip install -q transformers datasets accelerate scikit-learn matplotlib pandas

# 2. Clone your branch from GitHub
!git clone -b feature/c3-nlp-baseline-models https://github.com/SupunPrabodha/J26-SE-338.git

# 3. Navigate to notebook directory
%cd J26-SE-338/research/c3-model-experiments/notebooks

# 4. Standard Python libraries (only what is actually used)
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 5. Connect local src/ folder
sys.path.append('../src')

# 6. Verify GPU
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU.")

# 7. Import our research modules
from data_loader import load_dreaddit_dataset, load_goemotions_dataset, EKMAN_LABELS
from transformer_trainer import train_transformer_candidate
from metrics import evaluate_predictions

## 1. Load Standardized 80/20 Dataset Splits

In [ ]:
dreaddit_train, dreaddit_test = load_dreaddit_dataset(data_dir="../data/dreaddit")
goemotions_train, goemotions_test = load_goemotions_dataset(data_dir="../data/goemotions", use_ekman=True)

dreaddit_targets = ["Non-Stress", "Stress"]
goemotions_targets = [EKMAN_LABELS[i] for i in range(len(EKMAN_LABELS))]

## 2. Train Candidates on Dreaddit (Stress Classification)

In [ ]:
results_dreaddit = {}

# 1. XLM-RoBERTa
results_dreaddit["xlm-roberta-base"] = train_transformer_candidate(
    model_name_or_path="xlm-roberta-base",
    train_df=dreaddit_train,
    test_df=dreaddit_test,
    num_labels=2,
    target_names=dreaddit_targets,
    dataset_name="Dreaddit",
    output_dir="../results/transformers",
    batch_size=16,
    num_epochs=3,
    learning_rate=2e-5
)

# 2. mBERT
results_dreaddit["bert-base-multilingual-cased"] = train_transformer_candidate(
    model_name_or_path="bert-base-multilingual-cased",
    train_df=dreaddit_train,
    test_df=dreaddit_test,
    num_labels=2,
    target_names=dreaddit_targets,
    dataset_name="Dreaddit",
    output_dir="../results/transformers",
    batch_size=16,
    num_epochs=3,
    learning_rate=2e-5
)

## 3. Train Candidates on GoEmotions (Emotion Classification)

In [ ]:
results_goemotions = {}

# 1. XLM-RoBERTa
results_goemotions["xlm-roberta-base"] = train_transformer_candidate(
    model_name_or_path="xlm-roberta-base",
    train_df=goemotions_train,
    test_df=goemotions_test,
    num_labels=len(goemotions_targets),
    target_names=goemotions_targets,
    dataset_name="GoEmotions",
    output_dir="../results/transformers",
    batch_size=32,
    num_epochs=3,
    learning_rate=3e-5
)

# 2. mBERT
results_goemotions["bert-base-multilingual-cased"] = train_transformer_candidate(
    model_name_or_path="bert-base-multilingual-cased",
    train_df=goemotions_train,
    test_df=goemotions_test,
    num_labels=len(goemotions_targets),
    target_names=goemotions_targets,
    dataset_name="GoEmotions",
    output_dir="../results/transformers",
    batch_size=32,
    num_epochs=3,
    learning_rate=3e-5
)

## 4. Compile Stage A Benchmark Results Table

In [ ]:
# Load baseline results from disk
with open("../results/baselines/baseline_summary.json", "r") as f:
    baseline_results = json.load(f)

comparison_records = []

# Add Dreaddit records
for m, met in baseline_results["dreaddit"].items():
    comparison_records.append({"Dataset": "Dreaddit", "Category": "Baseline", "Model": f"TF-IDF + {m}", "Accuracy": met["accuracy"], "Macro F1": met["f1_macro"], "Weighted F1": met["f1_weighted"], "ECE": met.get("expected_calibration_error", "N/A")})
for m, met in results_dreaddit.items():
    comparison_records.append({"Dataset": "Dreaddit", "Category": "Transformer Candidate", "Model": m, "Accuracy": met["accuracy"], "Macro F1": met["f1_macro"], "Weighted F1": met["f1_weighted"], "ECE": met.get("expected_calibration_error", "N/A")})

# Add GoEmotions records
for m, met in baseline_results["goemotions"].items():
    comparison_records.append({"Dataset": "GoEmotions", "Category": "Baseline", "Model": f"TF-IDF + {m}", "Accuracy": met["accuracy"], "Macro F1": met["f1_macro"], "Weighted F1": met["f1_weighted"], "ECE": met.get("expected_calibration_error", "N/A")})
for m, met in results_goemotions.items():
    comparison_records.append({"Dataset": "GoEmotions", "Category": "Transformer Candidate", "Model": m, "Accuracy": met["accuracy"], "Macro F1": met["f1_macro"], "Weighted F1": met["f1_weighted"], "ECE": met.get("expected_calibration_error", "N/A")})

df_comparison = pd.DataFrame(comparison_records)
df_comparison.to_csv("../results/stage_a_comparison_table.csv", index=False)
df_comparison

## 5. Comparative Visualization & Decision Justification
The bar chart below summarizes model performance across both datasets to provide empirical justification for the winning candidate carried into **Stage B**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pivot_f1 = df_comparison.pivot(index='Model', columns='Dataset', values='Macro F1')
pivot_f1.plot(kind='bar', ax=ax, rot=25, colormap='viridis')
plt.title('Stage A: Baseline vs Multilingual Transformer Macro F1 Comparison')
plt.ylabel('Macro F1 Score')
plt.ylim(0, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Dataset')
plt.tight_layout()
plt.savefig('../results/stage_a_macro_f1_comparison.png', dpi=300)
plt.show()